## Targeted Experiments
This notebooks performs some selected targeted experiments on dataset to check how differetn features or techniques affect final model performance

In [1]:
import pandas as pd
import json
import numpy as np

from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR, NuSVR

from sklearn.model_selection import train_test_split

from helpers.modeling import (
    identify_column_types,
    create_preprocessor,
    evaluate_model,
    prepare_data,
    run_grid_search
)

from tqdm import tqdm


with open('results.json', 'r') as f:
    results = json.load(f)

params = results['best_params']['recoded_50']

df = pd.read_csv("../Datasets/processed/UHPC_dataset/semantic_recoding_features_50_with_publications.csv", index_col=0)

Remove worst cement types

In [2]:
df_cement_type_dropped = df[~df['cement_type'].isin(['OPC_53', 'Unknown'])]
print(df_cement_type_dropped.shape)

X = df_cement_type_dropped.drop(columns=['cs_28d'])
y = df_cement_type_dropped['cs_28d']

numerical_cols, one_hot_columns, k_fold_columns = identify_column_types(X)

preprocessor = create_preprocessor(numerical_cols, one_hot_columns, k_fold_columns,
                                   handle_unknown='ignore')

(1962, 34)


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42
)

knn_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', KNeighborsRegressor())
])
knn_pipeline.set_params(**params['knn'])

knn_pipeline.fit(X_train, y_train)

y_pred = knn_pipeline.predict(X_test)

print("KNN RESULTS WHEN REMOVING TOP TWO WORST PERFORMING CEMENT TYPES")
train_metrics = evaluate_model(y_train, knn_pipeline.predict(X_train), 'Training')
test_metrics = evaluate_model(y_test, y_pred, 'Test')

KNN RESULTS WHEN REMOVING TOP TWO WORST PERFORMING CEMENT TYPES

TRAINING SET PERFORMANCE
RMSE: 4.7792
MAE: 2.9968
R2: 0.9818
Correlation: 0.9910
Mean_Residual: -0.1155
N: 1373

TEST SET PERFORMANCE
RMSE: 14.7212
MAE: 10.2966
R2: 0.8211
Correlation: 0.9067
Mean_Residual: -0.4018
N: 589


In [4]:
svr_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', SVR())
])
svr_pipeline.set_params(**params['svr'])

svr_pipeline.fit(X_train, y_train)

y_pred = svr_pipeline.predict(X_test)

print("SVR RESULTS WHEN REMOVING TOP TWO WORST PERFORMING CEMENT TYPES")
train_metrics = evaluate_model(y_train, svr_pipeline.predict(X_train), 'Training')
test_metrics = evaluate_model(y_test, y_pred, 'Test')

SVR RESULTS WHEN REMOVING TOP TWO WORST PERFORMING CEMENT TYPES

TRAINING SET PERFORMANCE
RMSE: 7.6099
MAE: 5.0984
R2: 0.9538
Correlation: 0.9768
Mean_Residual: -0.2984
N: 1373

TEST SET PERFORMANCE
RMSE: 12.7557
MAE: 8.6630
R2: 0.8657
Correlation: 0.9307
Mean_Residual: -0.4990
N: 589


In [5]:
nusvr_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', NuSVR())
])
nusvr_pipeline.set_params(**params['nusvr'])

nusvr_pipeline.fit(X_train, y_train)

y_pred = nusvr_pipeline.predict(X_test)

print("NuSVR RESULTS WHEN REMOVING TOP TWO WORST PERFORMING CEMENT TYPES")
train_metrics = evaluate_model(y_train, nusvr_pipeline.predict(X_train), 'Training')
test_metrics = evaluate_model(y_test, y_pred, 'Test')

NuSVR RESULTS WHEN REMOVING TOP TWO WORST PERFORMING CEMENT TYPES

TRAINING SET PERFORMANCE
RMSE: 7.5845
MAE: 4.8987
R2: 0.9541
Correlation: 0.9771
Mean_Residual: -0.1776
N: 1373

TEST SET PERFORMANCE
RMSE: 12.7072
MAE: 8.6209
R2: 0.8667
Correlation: 0.9313
Mean_Residual: -0.5176
N: 589


## Remove all fibre columns

In [84]:
fibre_cols = ['fiber1_type', 'fiber1_amount', 'fiber1_length', 'fiber1_diameter', 'fiber2_type', 'fiber2_amount']

df_no_fibre = df.drop(columns=fibre_cols)
print(df_no_fibre.shape)

X_f = df_no_fibre.drop(columns=['cs_28d'])
y_f = df_no_fibre['cs_28d']

numerical_cols_f, one_hot_cols_f, k_fold_cols_f = identify_column_types(X_f)

preprocessor_f = create_preprocessor(numerical_cols_f, one_hot_cols_f, k_fold_cols_f,
                                     handle_unknown='ignore')

X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_f, y_f, test_size=0.30, random_state=42
)

(2073, 28)


In [85]:
knn_pipeline_f = Pipeline([
    ('preprocessor', preprocessor_f),
    ('model', KNeighborsRegressor())
])
knn_pipeline_f.set_params(**params['knn'])

knn_pipeline_f.fit(X_train_f, y_train_f)

y_pred = knn_pipeline_f.predict(X_test_f)

print("KNN RESULTS WHEN REMOVING ALL FIBRE COLUMNS")
train_metrics = evaluate_model(y_train_f, knn_pipeline_f.predict(X_train_f), 'Training')
test_metrics = evaluate_model(y_test_f, y_pred, 'Test')

KNN RESULTS WHEN REMOVING ALL FIBRE COLUMNS

TRAINING SET PERFORMANCE
RMSE: 12.3034
MAE: 6.8584
R2: 0.8864
Correlation: 0.9418
Mean_Residual: -0.7087
N: 1451

TEST SET PERFORMANCE
RMSE: 20.9361
MAE: 14.7001
R2: 0.6619
Correlation: 0.8263
Mean_Residual: -0.8912
N: 622


In [86]:
svr_pipeline_f = Pipeline([
    ('preprocessor', preprocessor_f),
    ('model', SVR())
])
svr_pipeline_f.set_params(**params['svr'])

svr_pipeline_f.fit(X_train_f, y_train_f)

y_pred = svr_pipeline_f.predict(X_test_f)

print("SVR RESULTS WHEN REMOVING ALL FIBRE COLUMNS")
train_metrics = evaluate_model(y_train_f, svr_pipeline_f.predict(X_train_f), 'Training')
test_metrics = evaluate_model(y_test_f, y_pred, 'Test')

SVR RESULTS WHEN REMOVING ALL FIBRE COLUMNS

TRAINING SET PERFORMANCE
RMSE: 14.0187
MAE: 8.5083
R2: 0.8525
Correlation: 0.9233
Mean_Residual: 0.1487
N: 1451

TEST SET PERFORMANCE
RMSE: 18.7291
MAE: 13.2686
R2: 0.7294
Correlation: 0.8565
Mean_Residual: -0.4099
N: 622


In [87]:
nusvr_pipeline_f = Pipeline([
    ('preprocessor', preprocessor_f),
    ('model', NuSVR())
])
nusvr_pipeline_f.set_params(**params['nusvr'])

nusvr_pipeline_f.fit(X_train_f, y_train_f)

y_pred = nusvr_pipeline_f.predict(X_test_f)

print("NuSVR RESULTS WHEN REMOVING ALL FIBRE COLUMNS")
train_metrics = evaluate_model(y_train_f, nusvr_pipeline_f.predict(X_train_f), 'Training')
test_metrics = evaluate_model(y_test_f, y_pred, 'Test')

NuSVR RESULTS WHEN REMOVING ALL FIBRE COLUMNS

TRAINING SET PERFORMANCE
RMSE: 14.0556
MAE: 8.8287
R2: 0.8517
Correlation: 0.9230
Mean_Residual: 0.3605
N: 1451

TEST SET PERFORMANCE
RMSE: 18.9497
MAE: 13.6420
R2: 0.7230
Correlation: 0.8531
Mean_Residual: -0.2221
N: 622


## Test on dataset without semantic recoding

In [88]:
df_raw = pd.read_csv("../Datasets/processed/UHPC_dataset/initial_cleaned.csv", index_col=0)
print(df_raw.shape)

X_r = df_raw.drop(columns=['cs_28d'])
y_r = df_raw['cs_28d']

numerical_cols_r, one_hot_cols_r, k_fold_cols_r = identify_column_types(X_r)

preprocessor_r = create_preprocessor(numerical_cols_r, one_hot_cols_r, k_fold_cols_r,
                                     handle_unknown='ignore')

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_r, y_r, test_size=0.30, random_state=42
)

(2073, 43)


In [89]:
knn_pipeline_r = Pipeline([
    ('preprocessor', preprocessor_r),
    ('model', KNeighborsRegressor())
])
knn_pipeline_r.set_params(**params['knn'])

knn_pipeline_r.fit(X_train_r, y_train_r)

y_pred = knn_pipeline_r.predict(X_test_r)

print("KNN RESULTS — NO SEMANTIC RECODING")
train_metrics = evaluate_model(y_train_r, knn_pipeline_r.predict(X_train_r), 'Training')
test_metrics = evaluate_model(y_test_r, y_pred, 'Test')

KNN RESULTS — NO SEMANTIC RECODING

TRAINING SET PERFORMANCE
RMSE: 5.4976
MAE: 3.2851
R2: 0.9773
Correlation: 0.9888
Mean_Residual: -0.2348
N: 1451

TEST SET PERFORMANCE
RMSE: 16.1639
MAE: 11.4127
R2: 0.7985
Correlation: 0.8952
Mean_Residual: -0.3765
N: 622


In [90]:
svr_pipeline_r = Pipeline([
    ('preprocessor', preprocessor_r),
    ('model', SVR())
])
svr_pipeline_r.set_params(**params['svr'])

svr_pipeline_r.fit(X_train_r, y_train_r)

y_pred = svr_pipeline_r.predict(X_test_r)

print("SVR RESULTS — NO SEMANTIC RECODING")
train_metrics = evaluate_model(y_train_r, svr_pipeline_r.predict(X_train_r), 'Training')
test_metrics = evaluate_model(y_test_r, y_pred, 'Test')

SVR RESULTS — NO SEMANTIC RECODING

TRAINING SET PERFORMANCE
RMSE: 8.6079
MAE: 5.4908
R2: 0.9444
Correlation: 0.9721
Mean_Residual: 0.0071
N: 1451

TEST SET PERFORMANCE
RMSE: 14.2415
MAE: 9.8118
R2: 0.8436
Correlation: 0.9191
Mean_Residual: 0.0630
N: 622


In [91]:
nusvr_pipeline_r = Pipeline([
    ('preprocessor', preprocessor_r),
    ('model', NuSVR())
])
nusvr_pipeline_r.set_params(**params['nusvr'])

nusvr_pipeline_r.fit(X_train_r, y_train_r)

y_pred = nusvr_pipeline_r.predict(X_test_r)

print("NuSVR RESULTS — NO SEMANTIC RECODING")
train_metrics = evaluate_model(y_train_r, nusvr_pipeline_r.predict(X_train_r), 'Training')
test_metrics = evaluate_model(y_test_r, y_pred, 'Test')

NuSVR RESULTS — NO SEMANTIC RECODING

TRAINING SET PERFORMANCE
RMSE: 8.4888
MAE: 5.2963
R2: 0.9459
Correlation: 0.9728
Mean_Residual: 0.1214
N: 1451

TEST SET PERFORMANCE
RMSE: 14.1183
MAE: 9.7396
R2: 0.8463
Correlation: 0.9206
Mean_Residual: 0.0654
N: 622
